### Gold Layer: Payment Weekly Summary (Agg Table for Power BI)
### 
- Pre-aggregated weekly summary joining FactTransactions with ALL dimension tables.
- Designed to replace DirectQuery scans on the full fact table for Power BI report visuals.
- **Source**: gold.FactTransactions + all Gold Dim tables
- **Grain**: Week × key dimension attributes
- **Target**: gold.PaymentWeeklySummary


In [0]:
# Import modules
import os, sys
notebook_path = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.append(f"/Workspace{os.sep.join(notebook_path.partition('notebooks')[:2])}")

from delta.tables import *
from pyspark.sql.functions import *
from Common.DataLakeURIs import *
from Common.DeltaTableHelper import *
from Common.Telemetry import *
from PaymentTransactions.PaymentTransactions_TableNames import *

## Create target table schema

In [0]:
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS gold.PaymentWeeklySummary (
        -- =====================================================
        -- TIME GRAIN
        -- =====================================================
        WeekStartDate                       DATE        NOT NULL,
        YearMonth                           INT         NOT NULL,

        -- =====================================================
        -- DimPayment (transaction profile)
        -- =====================================================
        TransactionTypeGroup                STRING,
        TransactionType                     STRING,
        PaymentLayer                        STRING,
        PaymentPartner                      STRING,
        ProviderName                        STRING,
        TransactionStatus                   STRING,
        ConsumerOrCommercial                STRING,
        CustomerOrMerchantInitiated         STRING,
        IsRecurring                         BOOLEAN,
        IsAuthTerminalState                 BOOLEAN,
        IsAuthApproval                      BOOLEAN,
        IsTransactionAbandoned              BOOLEAN,
        IsPayNow                            BOOLEAN,
        IsMicrosoftDecline                  BOOLEAN,
        PaymentNetworkName                  STRING,

        -- =====================================================
        -- DimGeo
        -- =====================================================
        Country                             STRING,
        Country_a3                          STRING,
        Region                              STRING,
        Currency                            STRING,
        Currency_a3                         STRING,

        -- =====================================================
        -- DimBIN
        -- =====================================================
        BinCardType                         STRING,
        BinCardProduct                      STRING,
        IsReloadablePrepaid                 STRING,
        IssuerName                          STRING,
        IssuerCountry                       STRING,

        -- =====================================================
        -- DimResponseCode
        -- =====================================================
        ResponseCode                        STRING,
        ResponseCodeDetails                 STRING,
        ChurnCategory                       STRING,
        MerchantAdviceCode                  STRING,

        -- =====================================================
        -- DimBilling
        -- =====================================================
        BillingPartner                      STRING,
        SubscriptionTerm                    STRING,
        SubscriptionPaymentClass            STRING,

        -- =====================================================
        -- DimMerchant
        -- =====================================================
        MerchantCountry                     STRING,
        MerchantCategoryCode                STRING,
        SellerOfRecord                      STRING,

        -- =====================================================
        -- DimAuthentication
        -- =====================================================
        PSD2ReferenceType                   STRING,
        PSD2Scenario                        STRING,

        -- =====================================================
        -- DimChargeback
        -- =====================================================
        ChargebackAttempt                   INT,
        IsLastCBEventInCBEventGroup         BOOLEAN,

        -- =====================================================
        -- DimCoBranded
        -- =====================================================
        IsXboxCoBrandedCard                 BOOLEAN,

        -- =====================================================
        -- DimNetworkToken
        -- =====================================================
        NetworkToken                        STRING,

        -- =====================================================
        -- DimDunningNew (ByCycle)
        -- =====================================================
        IsFirstDunAttemptByCycle            BOOLEAN,
        IsDunningCycle                      BOOLEAN,

        -- =====================================================
        -- DimAccountUpdater
        -- =====================================================
        RealtimeAUStatusFromProvider        STRING,

        -- =====================================================
        -- PRE-AGGREGATED MEASURES
        -- =====================================================
        TransactionCount                    BIGINT      NOT NULL,
        ApprovalCount                       BIGINT      NOT NULL,
        ApprovalAmountUSD                   DECIMAL(38,9) NOT NULL,
        TerminalStateCount                  BIGINT      NOT NULL,
        TerminalStateAmountUSD              DECIMAL(38,9) NOT NULL,
        DeclineCount_FirstAttempt           BIGINT      NOT NULL,
        DeclineCount_AllAttempts            BIGINT      NOT NULL,
        AbandonedCount                      BIGINT      NOT NULL,
        TotalAmountUSD                      DECIMAL(38,9) NOT NULL,
        TotalAmountReceivedUSD              DECIMAL(38,9) NOT NULL,

        -- =====================================================
        -- METADATA
        -- =====================================================
        AggregationTimestamp                TIMESTAMP   NOT NULL
    )
    USING DELTA
    PARTITIONED BY (YearMonth)
    LOCATION '{DataLakeURIs.main_root}/gold/PaymentWeeklySummary'
    TBLPROPERTIES (
        'quality' = 'gold',
        'description' = 'Weekly pre-aggregated payment metrics for Power BI. Joins FactTransactions with all dimension tables.',
        'subject_area' = '{PAYMENT_TRANSACTION_SUBJECT_AREA}'
    )
''')

## Load FactTransactions + Join All Dimensions

In [0]:
df_fact = spark.read.table(FACT_TRANSACTIONS_TABLE_NAME)

# -- DimPayment --
df_dim_payment = spark.read.table(DIM_PAYMENT_TABLE_NAME).alias("dp")
df_fact = df_fact.join(
    df_dim_payment,
    df_fact.PaymentId == df_dim_payment.Id,
    "left"
)

# -- DimGeo --
df_dim_geo = spark.read.table(DIM_GEO_TABLE_NAME).alias("dg")
df_fact = df_fact.join(
    df_dim_geo,
    df_fact.GeoId == df_dim_geo.Id,
    "left"
)

# -- DimBIN --
df_dim_bin = spark.read.table(DIM_BIN_TABLE_NAME).alias("db")
df_fact = df_fact.join(
    df_dim_bin,
    df_fact.BinId == df_dim_bin.BinId,
    "left"
)

# -- DimResponseCode --
df_dim_rc = spark.read.table(DIM_RESPONSECODE_TABLE_NAME).alias("drc")
df_fact = df_fact.join(
    df_dim_rc,
    df_fact.ResponseCodeId == df_dim_rc.ResponseCodeId,
    "left"
)

# -- DimBilling --
df_dim_billing = spark.read.table(DIM_BILLING_TABLE_NAME).alias("dbl")
df_fact = df_fact.join(
    df_dim_billing,
    df_fact.BillingId == df_dim_billing.BillingId,
    "left"
)

# -- DimMerchant --
df_dim_merchant = spark.read.table(DIM_MERCHANT_TABLE_NAME).alias("dm")
df_fact = df_fact.join(
    df_dim_merchant,
    df_fact.MerchantId == df_dim_merchant.Id,
    "left"
)

# -- DimAuthentication --
df_dim_auth = spark.read.table(DIM_AUTHENTICATION_TABLE_NAME).alias("da")
df_fact = df_fact.join(
    df_dim_auth,
    df_fact.AuthenticationId == df_dim_auth.Id,
    "left"
)

# -- DimChargeback --
df_dim_cb = spark.read.table(DIM_CHARGEBACK_TABLE_NAME).alias("dcb")
df_fact = df_fact.join(
    df_dim_cb,
    df_fact.ChargebackId == df_dim_cb.Id,
    "left"
)

# -- DimCoBranded --
df_dim_cobrand = spark.read.table(DIM_COBRANDED_TABLE_NAME).alias("dco")
df_fact = df_fact.join(
    df_dim_cobrand,
    df_fact.CoBrandedId == df_dim_cobrand.CoBrandedId,
    "left"
)

# -- DimNetworkToken --
df_dim_nt = spark.read.table(DIM_NETWORKTOKEN_TABLE_NAME).alias("dnt")
df_fact = df_fact.join(
    df_dim_nt,
    df_fact.NetworkTokenId == df_dim_nt.Id,
    "left"
)

# -- DimDunningNew (ByCycle) --
df_dim_dunning = spark.read.table(DIM_DUNNING_BYCYCLE_TABLE_NAME).alias("dd")
df_fact = df_fact.join(
    df_dim_dunning,
    df_fact.DunningByCycleId == df_dim_dunning.DunningByCycleId,
    "left"
)

# -- DimAccountUpdater --
df_dim_au = spark.read.table(DIM_ACCOUNT_UPDATER_TABLE_NAME).alias("dau")
df_fact = df_fact.join(
    df_dim_au,
    df_fact.AccountUpdaterId == df_dim_au.Id,
    "left"
)

print(f"Joined fact table columns: {len(df_fact.columns)}")

## Aggregate at Weekly Grain

In [0]:
# Define weekly grain
df_with_week = df_fact.withColumn("WeekStartDate", date_trunc("week", col("Date")))

# Group by columns - all dimension attributes used by report slicers + key analysis dimensions
group_by_cols = [
    # Time
    "WeekStartDate",
    col("Date").alias("_date_for_ym"),  # for YearMonth partition

    # DimPayment
    col("dp.TransactionTypeGroup"),
    col("dp.TransactionType"),
    col("dp.PaymentLayer"),
    col("dp.PaymentPartner"),
    col("dp.ProviderName"),
    col("dp.TransactionStatus"),
    col("dp.ConsumerOrCommercial"),
    col("dp.CustomerOrMerchantInitiated"),
    col("dp.IsRecurring"),
    col("dp.IsAuthTerminalState"),
    col("dp.IsAuthApproval"),
    col("dp.IsTransactionAbandoned"),
    col("dp.IsPayNow"),
    col("dp.IsMicrosoftDecline"),
    col("dp.PaymentNetworkName"),

    # DimGeo
    col("dg.Country"),
    col("dg.Country_a3"),
    col("dg.Region"),
    col("dg.Currency"),
    col("dg.Currency_a3"),

    # DimBIN
    col("db.BinCardType"),
    col("db.BinCardProduct"),
    col("db.IsReloadablePrepaid"),
    col("db.IssuerName"),
    col("db.IssuerCountry"),

    # DimResponseCode
    col("drc.ResponseCode"),
    col("drc.ResponseCodeDetails"),
    col("drc.ChurnCategory"),
    col("drc.MerchantAdviceCode"),

    # DimBilling
    col("dbl.BillingPartner"),
    col("dbl.SubscriptionTerm"),
    col("dbl.SubscriptionPaymentClass"),

    # DimMerchant
    col("dm.MerchantCountry"),
    col("dm.MerchantCategoryCode"),
    col("dm.SellerOfRecord"),

    # DimAuthentication
    col("da.PSD2ReferenceType"),
    col("da.PSD2Scenario"),

    # DimChargeback
    col("dcb.ChargebackAttempt"),
    col("dcb.IsLastCBEventInCBEventGroup"),

    # DimCoBranded
    col("dco.IsXboxCoBrandedCard"),

    # DimNetworkToken
    col("dnt.NetworkToken"),

    # DimDunningNew
    col("dd.IsFirstDunAttemptByCycle"),
    col("dd.IsDunningCycle"),

    # DimAccountUpdater
    col("dau.RealtimeAUStatusFromProvider"),
]

# Build aggregations
df_agg = (
    df_with_week
    .groupBy(
        "WeekStartDate",
        # DimPayment
        col("dp.TransactionTypeGroup").alias("TransactionTypeGroup"),
        col("dp.TransactionType").alias("TransactionType"),
        col("dp.PaymentLayer").alias("PaymentLayer"),
        col("dp.PaymentPartner").alias("PaymentPartner"),
        col("dp.ProviderName").alias("ProviderName"),
        col("dp.TransactionStatus").alias("TransactionStatus"),
        col("dp.ConsumerOrCommercial").alias("ConsumerOrCommercial"),
        col("dp.CustomerOrMerchantInitiated").alias("CustomerOrMerchantInitiated"),
        col("dp.IsRecurring").alias("IsRecurring"),
        col("dp.IsAuthTerminalState").alias("IsAuthTerminalState"),
        col("dp.IsAuthApproval").alias("IsAuthApproval"),
        col("dp.IsTransactionAbandoned").alias("IsTransactionAbandoned"),
        col("dp.IsPayNow").alias("IsPayNow"),
        col("dp.IsMicrosoftDecline").alias("IsMicrosoftDecline"),
        col("dp.PaymentNetworkName").alias("PaymentNetworkName"),
        # DimGeo
        col("dg.Country").alias("Country"),
        col("dg.Country_a3").alias("Country_a3"),
        col("dg.Region").alias("Region"),
        col("dg.Currency").alias("Currency"),
        col("dg.Currency_a3").alias("Currency_a3"),
        # DimBIN
        col("db.BinCardType").alias("BinCardType"),
        col("db.BinCardProduct").alias("BinCardProduct"),
        col("db.IsReloadablePrepaid").alias("IsReloadablePrepaid"),
        col("db.IssuerName").alias("IssuerName"),
        col("db.IssuerCountry").alias("IssuerCountry"),
        # DimResponseCode
        col("drc.ResponseCode").alias("ResponseCode"),
        col("drc.ResponseCodeDetails").alias("ResponseCodeDetails"),
        col("drc.ChurnCategory").alias("ChurnCategory"),
        col("drc.MerchantAdviceCode").alias("MerchantAdviceCode"),
        # DimBilling
        col("dbl.BillingPartner").alias("BillingPartner"),
        col("dbl.SubscriptionTerm").alias("SubscriptionTerm"),
        col("dbl.SubscriptionPaymentClass").alias("SubscriptionPaymentClass"),
        # DimMerchant
        col("dm.MerchantCountry").alias("MerchantCountry"),
        col("dm.MerchantCategoryCode").alias("MerchantCategoryCode"),
        col("dm.SellerOfRecord").alias("SellerOfRecord"),
        # DimAuthentication
        col("da.PSD2ReferenceType").alias("PSD2ReferenceType"),
        col("da.PSD2Scenario").alias("PSD2Scenario"),
        # DimChargeback
        col("dcb.ChargebackAttempt").alias("ChargebackAttempt"),
        col("dcb.IsLastCBEventInCBEventGroup").alias("IsLastCBEventInCBEventGroup"),
        # DimCoBranded
        col("dco.IsXboxCoBrandedCard").alias("IsXboxCoBrandedCard"),
        # DimNetworkToken
        col("dnt.NetworkToken").alias("NetworkToken"),
        # DimDunningNew
        col("dd.IsFirstDunAttemptByCycle").alias("IsFirstDunAttemptByCycle"),
        col("dd.IsDunningCycle").alias("IsDunningCycle"),
        # DimAccountUpdater
        col("dau.RealtimeAUStatusFromProvider").alias("RealtimeAUStatusFromProvider"),
    )
    .agg(
        # Transaction count (from fact table's TransactionCount column)
        sum("TransactionCount").alias("TransactionCount"),

        # Approval metrics
        sum(
            when(col("dp.IsAuthApproval") == True, col("TransactionCount")).otherwise(0)
        ).alias("ApprovalCount"),

        sum(
            when(col("dp.IsAuthApproval") == True, col("AmountUSD")).otherwise(lit(0))
        ).alias("ApprovalAmountUSD"),

        # Terminal state (denominator for approval rate)
        sum(
            when(col("dp.IsAuthTerminalState") == True, col("TransactionCount")).otherwise(0)
        ).alias("TerminalStateCount"),

        sum(
            when(col("dp.IsAuthTerminalState") == True, col("AmountUSD")).otherwise(lit(0))
        ).alias("TerminalStateAmountUSD"),

        # Decline metrics
        sum(
            when(
                (col("dp.IsAuthApproval") == False) &
                (col("dp.IsAuthTerminalState") == True),
                col("TransactionCount")
            ).otherwise(0)
        ).alias("DeclineCount_AllAttempts"),

        # Abandoned
        sum(
            when(col("dp.IsTransactionAbandoned") == True, col("TransactionCount")).otherwise(0)
        ).alias("AbandonedCount"),

        # Dollar totals
        sum("AmountUSD").alias("TotalAmountUSD")
        # sum(col("AmountReceivedUSD")).alias("TotalAmountReceivedUSD"),
    )
)

# Add YearMonth partition and metadata
df_agg = (
    df_agg
    .withColumn("YearMonth",
        (year("WeekStartDate") * 100 + month("WeekStartDate")).cast("int"))
    .withColumn("AggregationTimestamp", current_timestamp())
    # DeclineCount_FirstAttempt needs DynamicRetryAttempt which is on the source fact
    # For now, set equal to AllAttempts; refine in v2 if needed
    .withColumn("DeclineCount_FirstAttempt", col("DeclineCount_AllAttempts"))
)

In [0]:
## Write to Delta Table (Full Rebuild)

row_count = df_agg.count()
print(f"Aggregation result: {row_count:,} rows")

(
    df_agg
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("YearMonth")
    .saveAsTable("gold.PaymentWeeklySummary")
)

print(f"Written {row_count:,} rows to gold.PaymentWeeklySummary")

## Optimize & Verify

In [0]:
 %sql
 OPTIMIZE gold.PaymentWeeklySummary
 ZORDER BY (WeekStartDate, TransactionTypeGroup, Country_a3, ProviderName);

In [0]:
%sql
--Verification: row count, date range, dimensions
SELECT
     COUNT(*)                        AS total_rows,
     SUM(TransactionCount)           AS total_transactions,
     SUM(ApprovalCount)              AS total_approvals,
     SUM(ApprovalAmountUSD)          AS total_approval_usd,
     MIN(WeekStartDate)              AS earliest_week,
     MAX(WeekStartDate)              AS latest_week,
     COUNT(DISTINCT WeekStartDate)   AS distinct_weeks,
     COUNT(DISTINCT Country_a3)      AS distinct_countries,
     COUNT(DISTINCT ProviderName)    AS distinct_providers,
     COUNT(DISTINCT ResponseCodeDetails) AS distinct_response_codes,
     COUNT(DISTINCT BinCardType)     AS distinct_card_types,
     COUNT(DISTINCT IssuerName)      AS distinct_issuers
 FROM gold.PaymentWeeklySummary

In [0]:
%sql
-- Cross-check: compare totals against FactTransactions
SELECT 
    'FactTransactions' AS source,
    SUM(TransactionCount) AS total_txn,
    SUM(AmountUSD) AS total_usd
FROM gold.Fact_Transactions
UNION ALL
SELECT 
    'WeeklySummary' AS source,
    SUM(TransactionCount) AS total_txn,
    SUM(TotalAmountUSD) AS total_usd
FROM gold.PaymentWeeklySummary

## Data Quality Checks

In [0]:
# Verify no data loss
fact_total = spark.sql(f"SELECT SUM(TransactionCount) AS t FROM {FACT_TRANSACTIONS_TABLE_NAME}").collect()[0]["t"]
agg_total = spark.sql("SELECT SUM(TransactionCount) AS t FROM gold.PaymentWeeklySummary").collect()[0]["t"]

if fact_total == agg_total:
    print(f"✅ PASS: Transaction counts match ({fact_total:,})")
else:
    diff = abs(fact_total - agg_total)
    pct = diff / fact_total * 100
    print(f"⚠️ WARN: Count mismatch. Fact={fact_total:,}, Agg={agg_total:,}, Diff={diff:,} ({pct:.4f}%)")
    if pct > 0.01:
        raise ValueError(f"Data loss exceeds 0.01% threshold: {pct:.4f}%")

# Verify approval counts
fact_approvals = spark.sql(f"""
    SELECT SUM(t.TransactionCount) AS t 
    FROM {FACT_TRANSACTIONS_TABLE_NAME} t
    JOIN {DIM_PAYMENT_TABLE_NAME} dp ON t.PaymentId = dp.Id
    WHERE dp.IsAuthApproval = true
""").collect()[0]["t"]
agg_approvals = spark.sql("SELECT SUM(ApprovalCount) AS t FROM gold.PaymentWeeklySummary").collect()[0]["t"]

if fact_approvals == agg_approvals:
    print(f"✅ PASS: Approval counts match ({fact_approvals:,})")
else:
    print(f"⚠️ WARN: Approval mismatch. Fact={fact_approvals:,}, Agg={agg_approvals:,}")